In [ ]:
import os
import json
import ipywidgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output

In [ ]:
#RESULTS_BASE_DIR = "../results_2025-05-12"
RESULTS_BASE_DIR = "../results"

## Load Results

In [ ]:
results_files = [f for f in os.listdir(RESULTS_BASE_DIR) if os.path.isdir(os.path.join(RESULTS_BASE_DIR, f))]
all_results = {}
active_results = {}
selected_configs = []
config_checkboxes = []
config_status_label = ipywidgets.HTML(value="")
config_selector_output = ipywidgets.Output()

def recompute_summary_from_run_files(result_dir):
    """
    Rebuild the same summary-like structure expected by this notebook:
    {config_name: [run_summary, ...], ...}
    using per-run files under <result_dir>/<config>/<env>/<run>.json.
    """
    reconstructed = {}

    if not os.path.isdir(result_dir):
        return reconstructed

    for cfg_name in sorted(os.listdir(result_dir)):
        cfg_dir = os.path.join(result_dir, cfg_name)
        if not os.path.isdir(cfg_dir):
            continue

        entries = []
        for env_name in sorted(os.listdir(cfg_dir)):
            env_dir = os.path.join(cfg_dir, env_name)
            if not os.path.isdir(env_dir):
                continue

            for fname in sorted(os.listdir(env_dir)):
                if not fname.lower().endswith(".json"):
                    continue

                run_path = os.path.join(env_dir, fname)
                try:
                    with open(run_path, "r", encoding="utf-8") as f:
                        payload = json.load(f)
                except (json.JSONDecodeError, OSError):
                    continue

                run_id_raw = payload.get("run", os.path.splitext(fname)[0])
                try:
                    run_id = int(run_id_raw)
                except (TypeError, ValueError):
                    try:
                        run_id = int(os.path.splitext(fname)[0])
                    except (TypeError, ValueError):
                        continue

                entry = {
                    "env": str(payload.get("env", env_name)),
                    "run": run_id,
                    "seed": int(payload.get("seed", -1)),
                    "max_steps": int(payload.get("max_steps", -1)),
                    "steps": int(payload.get("steps", payload.get("step_count", -1))),
                    "success": int(payload.get("success", 0)),
                    "reward": payload.get("reward", 0.0),
                    "history_file": os.path.relpath(run_path, start=RESULTS_BASE_DIR),
                }
                if isinstance(payload.get("config"), dict):
                    entry["config"] = payload.get("config")
                if "completed_at" in payload:
                    entry["completed_at"] = payload.get("completed_at")
                entries.append(entry)

        if entries:
            reconstructed[cfg_name] = sorted(
                entries,
                key=lambda item: (
                    str(item.get("env", "")),
                    int(item.get("run", 0)),
                ),
            )

    return reconstructed

def load_result_configurations(result_folder):
    filepath = os.path.join(RESULTS_BASE_DIR, result_folder, "summary.json")

    if os.path.exists(filepath):
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                loaded = json.load(f)
            if isinstance(loaded, dict) and loaded:
                return loaded, f"Loaded main summary: {filepath}"
        except (json.JSONDecodeError, OSError):
            pass

    fallback = recompute_summary_from_run_files(os.path.join(RESULTS_BASE_DIR, result_folder))
    return fallback, (
        "Main summary missing/invalid or empty. "
        "Recomputed from per-run JSON files."
    )

def recompute_active_results():
    global active_results, selected_configs
    selected_configs = [cb.description for cb in config_checkboxes if cb.value]
    active_results = {cfg: all_results[cfg] for cfg in selected_configs if cfg in all_results}

    total = len(config_checkboxes)
    selected = len(selected_configs)
    config_status_label.value = f"<b>Selected:</b> {selected}/{total} configurations"

def on_checkbox_change(change):
    if change["name"] == "value":
        recompute_active_results()

def set_all_config_checkboxes(value):
    for cb in config_checkboxes:
        cb.unobserve(on_checkbox_change, names="value")
        cb.value = value
        cb.observe(on_checkbox_change, names="value")
    recompute_active_results()

def build_config_selector_widget():
    global config_checkboxes
    config_names = sorted(all_results.keys())
    config_checkboxes = [
        ipywidgets.Checkbox(
            value=True,
            description=cfg,
            indent=False,
            layout=ipywidgets.Layout(width="auto"),
        )
        for cfg in config_names
    ]

    for cb in config_checkboxes:
        cb.observe(on_checkbox_change, names="value")

    select_all_button = ipywidgets.Button(description="Select all", button_style="success")
    clear_all_button = ipywidgets.Button(description="Clear all", button_style="warning")

    select_all_button.on_click(lambda _: set_all_config_checkboxes(True))
    clear_all_button.on_click(lambda _: set_all_config_checkboxes(False))

    config_grid = ipywidgets.GridBox(
        children=config_checkboxes,
        layout=ipywidgets.Layout(
            grid_template_columns="repeat(2, minmax(320px, 1fr))",
            grid_gap="4px 16px",
            max_height="320px",
            overflow_y="auto",
            border="1px solid #dddddd",
            padding="8px",
        ),
    )

    recompute_active_results()

    return ipywidgets.VBox([
        ipywidgets.HBox([select_all_button, clear_all_button]),
        config_status_label,
        config_grid,
    ])

def display_results(file):
    global all_results, active_results
    all_results, load_message = load_result_configurations(file)

    with config_selector_output:
        clear_output(wait=True)
        print(load_message)
        if not all_results:
            active_results = {}
            print("No configurations found in this result file.")
            return
        display(build_config_selector_widget())

ipywidgets.interact(
    display_results,
    file=ipywidgets.Dropdown(options=sorted(results_files), description="Result file"),
);
display(config_selector_output)

In [ ]:
# Treating history as part of the model identity
CHAT_MODEL_NAMES = ["GPT4.1", "GPT5.4", "Gemma4_2B", "DeepSeek_V4_Flash", "gemma-3-4b-it"]
CHAT_MODEL_NAMES = [f"{name}_with_history" for name in CHAT_MODEL_NAMES] + CHAT_MODEL_NAMES

def parse_model(cfg_name):
    # Reverse to match more-specific (history) variants first
    for model_name in reversed(CHAT_MODEL_NAMES):
        if model_name in cfg_name:
            return model_name
    return "Unknown"

In [ ]:
if not active_results:
    raise ValueError("No active experiment data to plot. Load a result file and select at least one configuration.")

## Plot Results

Choose the configurations in the selector above, then run the plot cells below.

### Per Configuration

Bar plot with one bar per configuration.

In [ ]:
config_names = list(active_results.keys())
success_rates = []
run_counts = []

In [ ]:
for cfg in config_names:
    runs = active_results.get(cfg, [])
    successes = [int(run.get("success", 0)) for run in runs]
    rate = 100 * np.mean(successes) if successes else np.nan
    success_rates.append(rate)
    run_counts.append(len(successes))

fig, ax = plt.subplots(figsize=(12, 5.5))
bars = ax.bar(config_names, success_rates)

for bar, rate, n_runs in zip(bars, success_rates, run_counts):
    if np.isnan(rate):
        continue
    ax.annotate(
        f"{rate:.1f}%\n(n={n_runs})",
        xy=(bar.get_x() + bar.get_width() / 2, rate),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title("Success Rate per Configuration")
ax.set_xlabel("Configuration")
ax.set_ylabel("Success Rate (%)")
ax.set_ylim(0, 115)
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

### Per Configuration X Environment

In [ ]:
# Collect successes by (configuration, environment)
success_by_cfg_env = {}
for cfg_name, runs in active_results.items():
    for run in runs:
        env = run["env"]
        success_by_cfg_env.setdefault((cfg_name, env), []).append(int(run["success"]))

if not success_by_cfg_env:
    raise ValueError("No experiment data found in active_results to plot.")

configs = sorted(active_results.keys())
envs = sorted({k[1] for k in success_by_cfg_env.keys()})

if len(envs) != 2:
    raise ValueError(f"Expected exactly 2 environments, found {len(envs)}: {envs}")

# Build one success-rate list per environment, aligned by configuration order.
rates_by_env = {env: [] for env in envs}
run_counts_by_env = {env: [] for env in envs}
for cfg in configs:
    for env in envs:
        vals = success_by_cfg_env.get((cfg, env), [])
        rate = 100 * np.mean(vals) if vals else np.nan
        rates_by_env[env].append(rate)
        run_counts_by_env[env].append(len(vals))

x = np.arange(len(configs))
bar_width = 0.36
offsets = [-bar_width / 2, bar_width / 2]

fig, ax = plt.subplots(figsize=(13, 5.8))
for env, offset in zip(envs, offsets):
    bars = ax.bar(x + offset, rates_by_env[env], width=bar_width, label=env)
    for b, n_runs in zip(bars, run_counts_by_env[env]):
        h = b.get_height()
        if not np.isnan(h):
            ax.annotate(f"{h:.1f}%\n(n={n_runs})",
                        xy=(b.get_x() + b.get_width() / 2, h),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha="center", va="bottom", fontsize=8)

ax.set_title("Success Rate per Configuration with Environment Pair")
ax.set_xlabel("Configuration")
ax.set_ylabel("Success Rate (%)")
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=25, ha="right")
ax.set_ylim(0, 120)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(title="Environment")
plt.tight_layout()
plt.show()

### Per Model (aggregate)

In [ ]:
# Gather successes for aggregate metrics
success_by_model = {}

for cfg_name, runs in active_results.items():
    model = parse_model(cfg_name)
    for run in runs:
        env = run["env"]
        success = int(run["success"])
        success_by_model.setdefault(model, []).append(success)

if not success_by_model:
    raise ValueError("No experiment data found in active_results to plot.")

# Aggregate success rate per model
model_labels = sorted(success_by_model.keys())
model_rates = [100 * np.mean(success_by_model[m]) for m in model_labels]

fig, ax = plt.subplots(figsize=(7, 4.8))
bars = ax.bar(model_labels, model_rates)
for b, rate in zip(bars, model_rates):
    ax.annotate(f"{rate:.1f}%",
                xy=(b.get_x() + b.get_width() / 2, rate),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=9)

ax.set_title("Aggregate Success Rate per Model")
ax.set_xlabel("Model")
ax.set_ylabel("Success Rate (%)")
ax.set_ylim(0, 115)
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

### Per Model × Environment

Grouped bar plot: for each model, one bar per environment.

In [ ]:
# Collect all successes by (model, environment), combining runs from all prompts
success_by_model_env = {}
for cfg_name, runs in active_results.items():
    model = parse_model(cfg_name)
    for run in runs:
        env = run["env"]
        success_by_model_env.setdefault((model, env), []).append(int(run["success"]))

if not success_by_model_env:
    raise ValueError("No experiment data found in active_results to plot.")

models = sorted({k[0] for k in success_by_model_env.keys()})
envs = sorted({k[1] for k in success_by_model_env.keys()})

if len(envs) != 2:
    raise ValueError(f"Expected exactly 2 environments, found {len(envs)}: {envs}")

# Build rates so each model has exactly two bars (one for each environment)
rates_by_env = {env: [] for env in envs}
for model in models:
    for env in envs:
        vals = success_by_model_env.get((model, env), [])
        rate = 100 * np.mean(vals) if vals else np.nan
        rates_by_env[env].append(rate)

x = np.arange(len(models))
bar_width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))
offsets = [-bar_width / 2, bar_width / 2]

for env, offset in zip(envs, offsets):
    bars = ax.bar(x + offset, rates_by_env[env], width=bar_width, label=env)
    for b in bars:
        h = b.get_height()
        if not np.isnan(h):
            ax.annotate(f"{h:.1f}%",
                        xy=(b.get_x() + b.get_width() / 2, h),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha="center", va="bottom", fontsize=9)

ax.set_title("Success Rate per Model with Two Environment Bars")
ax.set_xlabel("Model")
ax.set_ylabel("Success Rate (%)")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 115)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(title="Environment")
plt.tight_layout()
plt.show()

### Per Environment (aggregate)

In [ ]:
# Gather successes for aggregate metrics
success_by_env = {}

for cfg_name, runs in active_results.items():
    model = parse_model(cfg_name)
    for run in runs:
        env = run["env"]
        success = int(run["success"])
        success_by_env.setdefault(env, []).append(success)

if not success_by_env:
    raise ValueError("No experiment data found in active_results to plot.")

# Aggregate success rate per environment
env_labels = sorted(success_by_env.keys())
env_rates = [100 * np.mean(success_by_env[e]) for e in env_labels]

fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(env_labels, env_rates)
for b, rate in zip(bars, env_rates):
    ax.annotate(f"{rate:.1f}%",
                xy=(b.get_x() + b.get_width() / 2, rate),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=9)

ax.set_title("Aggregate Success Rate per Environment")
ax.set_xlabel("Environment")
ax.set_ylabel("Success Rate (%)")
ax.set_ylim(0, 115)
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()